# Pointing Game Evaluation for CAM Methods

This notebook evaluates various Class Activation Mapping (CAM) methods using the **Pointing Game** metric on VOC and COCO datasets. 

The Pointing Game checks if the maximum saliency point of a CAM falls within the ground truth bounding box of the target object. We specifically filter for samples where the model correctly classifies the object to ensure we are evaluating the explanation of a correct decision.

**Note:** The models are trained on multi-label classification tasks, but we use the detection versions of the datasets to access ground truth bounding boxes.

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import cv2
from tqdm import tqdm
from collections import defaultdict

# SpikingJelly imports
from spikingjelly.activation_based import neuron, surrogate, functional, layer

# Project imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.utils.config_utils import setup_config_dictionary
from src.utils.data_utils import get_datasets, get_dataset_transforms
from src.utils.model_utils import get_model
from src.utils.save_utils import load_model

# CAM imports
from src.cams.gradcam import SNNGradCAM
from src.cams.ablation_cam import AblationCAM
from src.cams.mc_ablation_cam import MCAblationCAM
from src.cams.sam import SpikeActivationMap

# Setup Device and Seed
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print(f"Running on device: {device}")

In [ ]:
# --- Configuration ---
config = setup_config_dictionary()

# Choose Dataset: 'voc' or 'coco'
DATASET_CHOICE = 'voc' 

if DATASET_CHOICE == 'voc':
    config["DATASET_NAME"] = "voc_multilabel" # For model loading/class mapping
    DETECTION_DATASET_NAME = "voc_detection"  # For bounding boxes
    config["MODEL_NAME"] = "spiking_vgg16_bn"       # Ensure this matches your trained model
    NUM_CLASSES = 20
elif DATASET_CHOICE == 'coco':
    config["DATASET_NAME"] = "coco_multilabel"
    DETECTION_DATASET_NAME = "coco"
    config["MODEL_NAME"] = "spiking_vgg16_bn"
    NUM_CLASSES = 80

config["TAU"] = 2.0
config["N_TIME_BINS"] = 8 # Use a small number for static image inference speed
INPUT_SIZE = (224, 224)   # Standard size for these models

print(f"Configured for {DATASET_CHOICE.upper()} dataset with model {config['MODEL_NAME']}")

In [ ]:
# --- Load Model ---
# We load the classification model.
spiking_neuron = neuron.IFNode

model = get_model(
    model_name=config["MODEL_NAME"],
    num_classes=NUM_CLASSES,
    in_channels=3,
    spiking_neuron=spiking_neuron,
    pretrained=False,
    detach_reset=False,
    surrogate_function=surrogate.ATan(),
)

functional.set_step_mode(model, 'm')
functional.set_backend(model, "torch", instance=spiking_neuron)
model.to(device)

# Load weights
# Note: Ensure the model path exists in your config["MODEL_SAVE_DIR"]
try:
    load_model(model, config["MODEL_SAVE_DIR"], config["DATASET_NAME"])
    print("Model weights loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load weights. Using random weights for demonstration. Error: {e}")

model.eval()
convs = [x for x in model.modules() if isinstance(x, layer.Conv2d)]
lifs = [x for x in model.modules() if isinstance(x, neuron.IFNode)]
print(f"Model has {len(convs)} Conv layers and {len(lifs)} Spiking layers.")

In [ ]:
# --- Load Datasets & Mappings ---

# 1. Load the Multilabel dataset briefly to get the class mapping (Name -> Index)
# We need this because the detection dataset might use strings or different IDs.
print(f"Loading {config['DATASET_NAME']} to extract class mappings...")
ds_cls, _ = get_datasets(config["DATASET_NAME"], n_time_bins=1)

if DATASET_CHOICE == 'voc':
    # VOC: classes are a list of strings
    class_names = ds_cls.classes
    class_to_idx = ds_cls.class_to_idx
    # Helper to map detection annotation name to model index
    def map_det_label_to_idx(label_name):
        return class_to_idx.get(label_name, -1)

elif DATASET_CHOICE == 'coco':
    # COCO: classes are strings, but detection returns category_ids
    class_names = ds_cls.classes
    # ds_cls.cat_id_to_idx maps COCO category ID to 0-79 index
    cat_id_to_idx = ds_cls.cat_id_to_idx
    def map_det_label_to_idx(cat_id):
        return cat_id_to_idx.get(cat_id, -1)

print(f"Class mapping loaded. {len(class_names)} classes found.")

# 2. Load the Detection dataset for evaluation
# We use the raw dataset to get bounding boxes, but we need to apply transforms manually 
# to the image to match the model input, while keeping track of scaling for boxes.
print(f"Loading {DETECTION_DATASET_NAME} for evaluation...")
# We load WITHOUT transforms initially to get original image sizes easily, 
# or we handle the resize logic explicitly. 
# The get_datasets function applies transforms. Let's use it but handle box scaling.
ds_det, _ = get_datasets(DETECTION_DATASET_NAME, n_time_bins=1)

print(f"Detection dataset loaded with {len(ds_det)} samples.")

In [ ]:
# --- CAM Generation Helpers (with SNN Workarounds) ---

def build_model_copy():
    """Creates a fresh copy of the model to avoid state corruption during CAM generation."""
    m = get_model(
        config["MODEL_NAME"],
        num_classes=NUM_CLASSES,
        in_channels=3,
        spiking_neuron=neuron.IFNode,
        pretrained=False,
        detach_reset=False,
        surrogate_function=surrogate.ATan(),
    )
    functional.set_step_mode(m, 'm')
    functional.set_backend(m, "torch", instance=neuron.IFNode)
    m.load_state_dict(model.state_dict(), strict=True)
    m.to(device)
    m.eval()
    return m

def get_target_layers(m, conv_idx, lif_idx):
    """Returns the specific conv layer and spiking layer."""
    convs = [x for x in m.modules() if isinstance(x, layer.Conv2d)]
    lifs = [x for x in m.modules() if isinstance(x, neuron.IFNode)]
    return convs[conv_idx], lifs[lif_idx]

def generate_cams(events, class_idx, conv_idx, lif_idx):
    """
    Generates CAMs using multiple methods for a specific class index and layer pair.
    events: (T, B, C, H, W)
    """
    cams = {}
    
    # 1. GradCAM
    gc_model = build_model_copy()
    target_conv, _ = get_target_layers(gc_model, conv_idx, lif_idx)
    
    # Workaround: Enable gradients
    for p in gc_model.parameters():
        p.requires_grad_(True)
    functional.reset_net(gc_model)
    
    grad_cam = SNNGradCAM(gc_model, target_conv)
    
    # Workaround: Train mode for grad flow, Eval for BN
    gc_model.train()
    for m in gc_model.modules():
        if isinstance(m, (layer.BatchNorm2d, layer.Dropout)):
            m.eval()
            
    gc_map = grad_cam.generate(events, class_idx=class_idx)
    cams['GradCAM'] = gc_map.detach().cpu() # (T, H, W)

    # 2. Ablation CAM
    ab_model = build_model_copy()
    target_conv, _ = get_target_layers(ab_model, conv_idx, lif_idx)
    functional.reset_net(ab_model)
    
    ab_cam = AblationCAM(ab_model, target_conv, channel_batch_size=8)
    ab_map = ab_cam.generate(events, class_idx=class_idx)
    cams['AblationCAM'] = ab_map.detach().cpu()

    # 3. MC Ablation CAM
    mc_model = build_model_copy()
    _, target_lif = get_target_layers(mc_model, conv_idx, lif_idx)
    functional.reset_net(mc_model)
    
    mc_cam = MCAblationCAM(mc_model, target_lif, channel_batch_size=8, k_trials=50)
    mc_map = mc_cam.generate(events, class_idx=class_idx)
    cams['MC_Ablation'] = mc_map.detach().cpu()

    # 4. SAM (Spike Activation Map)
    sam_model = build_model_copy()
    _, target_lif = get_target_layers(sam_model, conv_idx, lif_idx)
    functional.reset_net(sam_model)
    
    spike_map = SpikeActivationMap(sam_model, target_lif)
    sam_map = spike_map.generate(events)
    cams['SAM'] = sam_map.detach().cpu()

    return cams

In [ ]:
# --- Pointing Game Logic ---

def calculate_pointing_game(cam, bboxes):
    """
    Checks if the maximum point of the CAM lies inside any of the bounding boxes.
    cam: (H, W) numpy array or tensor (aggregated over time)
    bboxes: List of [xmin, ymin, xmax, ymax] (scaled to H, W)
    """
    if isinstance(cam, torch.Tensor):
        cam = cam.numpy()
        
    # Find coordinates of the maximum value
    # unravel_index returns (row, col) -> (y, x)
    y_max, x_max = np.unravel_index(cam.argmax(), cam.shape)
    
    hit = False
    for box in bboxes:
        xmin, ymin, xmax, ymax = box
        if xmin <= x_max <= xmax and ymin <= y_max <= ymax:
            hit = True
            break
            
    return hit, (x_max, y_max)

def get_scaled_bboxes(target, orig_size, new_size=(224, 224)):
    """
    Extracts bboxes for specific classes and scales them to the model input size.
    Returns a dict: {class_idx: [[x1, y1, x2, y2], ...]}
    """
    w_orig, h_orig = orig_size
    w_new, h_new = new_size
    scale_x = w_new / w_orig
    scale_y = h_new / h_orig
    
    boxes_by_class = defaultdict(list)
    
    if DATASET_CHOICE == 'voc':
        # VOC target is a dict
        objects = target['annotation']['object']
        if not isinstance(objects, list): objects = [objects]
        
        for obj in objects:
            name = obj['name']
            idx = map_det_label_to_idx(name)
            if idx == -1: continue
            
            bnd = obj['bndbox']
            # VOC is 1-based? Usually torchvision handles parsing, but raw dict is strings
            x1 = float(bnd['xmin']) * scale_x
            y1 = float(bnd['ymin']) * scale_y
            x2 = float(bnd['xmax']) * scale_x
            y2 = float(bnd['ymax']) * scale_y
            boxes_by_class[idx].append([x1, y1, x2, y2])
            
    elif DATASET_CHOICE == 'coco':
        # COCO target is a list of dicts
        for obj in target:
            cat_id = obj['category_id']
            idx = map_det_label_to_idx(cat_id)
            if idx == -1: continue
            
            # COCO bbox is [x, y, width, height]
            x, y, w, h = obj['bbox']
            x1 = x * scale_x
            y1 = y * scale_y
            x2 = (x + w) * scale_x
            y2 = (y + h) * scale_y
            boxes_by_class[idx].append([x1, y1, x2, y2])
            
    return boxes_by_class

In [ ]:
# --- Main Evaluation Loop ---

MAX_SAMPLES = 20 # Configurable number of samples to process
results = defaultdict(lambda: {'hits': 0, 'total': 0})
visual_samples = []

requested_layer_pairs = [
    (6, 6),   
    (9, 9),
    (12, 12),
]

print(f"Starting Pointing Game evaluation on {MAX_SAMPLES} samples...")

# We iterate by index to access raw dataset if needed, but ds_det returns transformed images.
# We need original size. 
# Workaround: For VOC, size is in annotation. For COCO, we might need to assume standard size or skip.
# Here we assume VOC structure for size extraction.

for i in tqdm(range(len(ds_det)), desc="Evaluating Samples", total=MAX_SAMPLES):
    if i >= MAX_SAMPLES: break
    
    img_tensor, target = ds_det[i]
    
    # 1. Get Original Size
    if DATASET_CHOICE == 'voc':
        orig_w = int(target['annotation']['size']['width'])
        orig_h = int(target['annotation']['size']['height'])
    elif DATASET_CHOICE == 'coco':
        orig_w, orig_h = 640, 400
        
    # 2. Prepare Input
    # (C, H, W) -> (T, B, C, H, W)
    input_tensor = img_tensor.unsqueeze(0).unsqueeze(0) # (1, 1, C, H, W)
    input_tensor = input_tensor.repeat(config["N_TIME_BINS"], 1, 1, 1, 1)
    input_tensor = input_tensor.to(device)
    
    # 3. Model Prediction
    functional.reset_net(model)
    with torch.no_grad():
        outputs = model(input_tensor) # (T, B, Classes)
        time_sum = outputs.sum(dim=0) # (B, Classes)
        probs = torch.sigmoid(time_sum)
        
    # 4. Identify Correct Classes
    # Get GT classes present in image
    gt_boxes_map = get_scaled_bboxes(target, (orig_w, orig_h), INPUT_SIZE)
    gt_classes = set(gt_boxes_map.keys())
    
    # Get Predicted classes (threshold 0.5)
    pred_classes = set(torch.where(probs[0] > 0.5)[0].cpu().numpy())
    
    # Intersection: Correctly classified objects
    correct_classes = gt_classes.intersection(pred_classes)
    
    if not correct_classes:
        continue
        
    for cls_idx in correct_classes:
        bboxes = gt_boxes_map[cls_idx]
        
        for (c_idx, l_idx) in requested_layer_pairs:
            layer_key = f"C{c_idx}_L{l_idx}"
            cams = generate_cams(input_tensor, cls_idx, c_idx, l_idx)
            
            sample_result = {
                'img': img_tensor,
                'class_name': class_names[cls_idx],
                'bboxes': bboxes,
                'cams': {},
                'layer_info': layer_key
            }
            
            for method_name, cam_data in cams.items():
                # Aggregate time (mean) -> (H, W)
                cam_agg = cam_data.mean(dim=0).numpy()
                
                # Resize CAM to 224x224 if it isn't already
                if cam_agg.shape != INPUT_SIZE:
                    cam_agg = cv2.resize(cam_agg, INPUT_SIZE)
                    
                hit, max_pt = calculate_pointing_game(cam_agg, bboxes)
                
                results[layer_key][method_name]['hits'] += int(hit)
                results[layer_key][method_name]['total'] += 1
                
                sample_result['cams'][method_name] = {
                    'map': cam_agg,
                    'hit': hit,
                    'max_pt': max_pt
                }
                
            if len(visual_samples) < 10:
                visual_samples.append(sample_result)

print("Evaluation complete.")

In [ ]:
data = []
for layer_key, method_res in results.items():
    for method, res in method_res.items():
        acc = res['hits'] / res['total'] if res['total'] > 0 else 0
        data.append({
            'Layer': layer_key,
            'Method': method,
            'Pointing Game Acc': f"{acc:.2%}",
            'Hits': res['hits'],
            'Total': res['total']
        })
    
df = pd.DataFrame(data)
print("Pointing Game Results (Correctly Classified Samples Only):")
display(df.sort_values(['Layer', 'Method']))

In [ ]:
# --- Visualization ---

def denormalize(tensor):
    """Un-normalizes ImageNet tensors for display."""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).numpy()
    img = std * img + mean
    return np.clip(img, 0, 1)

# Select a few interesting samples
samples_to_plot = visual_samples[:3] 

methods = list(results.keys())
n_methods = len(methods)

for i, sample in enumerate(samples_to_plot):
    fig, axes = plt.subplots(1, n_methods + 1, figsize=(4 * (n_methods + 1), 4))
    
    # 1. Original Image with GT Boxes
    ax_orig = axes[0]
    img_np = denormalize(sample['img'])
    ax_orig.imshow(img_np)
    ax_orig.set_title(f"GT: {sample['class_name']}")
    ax_orig.axis('off')
    
    # Draw GT boxes
    for box in sample['bboxes']:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='g', facecolor='none')
        ax_orig.add_patch(rect)
        
    # 2. CAMs
    for j, method in enumerate(methods):
        ax = axes[j+1]
        cam_data = sample['cams'][method]
        heatmap = cam_data['map']
        hit = cam_data['hit']
        max_pt = cam_data['max_pt']
        
        # Overlay
        ax.imshow(img_np)
        ax.imshow(heatmap, cmap='jet', alpha=0.5)
        
        # Draw Max Point
        ax.plot(max_pt[0], max_pt[1], 'rx', markersize=10, markeredgewidth=2)
        
        # Draw GT boxes again for reference
        for box in sample['bboxes']:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='w', facecolor='none', linestyle='--')
            ax.add_patch(rect)
            
        status = "HIT" if hit else "MISS"
        color = "green" if hit else "red"
        ax.set_title(f"{method}\n{status}", color=color, fontweight='bold')
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()